<h1 style="text-align: center;">Model Implementation</h1>
<h3 style="text-align: center;">Hotel Booking Cancellation Prediction</h3>

---

<h5 style="text-align: right;">By Beta Group</h5>

## Goal
Notebook ini menjelaskan bagaimana model final diimplementasikan pada data test, termasuk mapping hasil prediksi ke rekomendasi aksi bisnis, dan perhitungan cost-benefit dari penggunaan model.

## Alur
1. Global configuration & load model final + data test
2. Feature alignment check (memastikan tidak ada kolom yang tidak seharusnya masuk ke model, termasuk kolom yang pernah ditandai berisiko leakage)
3. Prediksi cancel + risk segmentation + mapping ke rekomendasi aksi (per level resiko)
4. Cost-benefit simulation dengan asumsi eksplisit soal efektivitas intervensi

In [1]:
import warnings
warnings.filterwarnings("ignore")

# **Section 0. Global Configuration**

In [2]:
MODEL_DIR = "../final_model/"
DATA_CSV = "../data/split/test.csv"

BEST_THRESHOLD = 0.7425999999999402

RISK_BINS = [-0.001, 0.3, BEST_THRESHOLD, 1.001]
RISK_LABELS = ["Low Risk", "Medium Risk", "High Risk"]

EMPLOYEE_COST_PER_MINUTE = 0.17
PHONE_COST_PER_MINUTE = 0.2

CONTACT_DURATION_MINUTES = 3

SUCCESS_RATES = [0.3, 0.5, 0.7]

ACTION_MAP = {
    "Low Risk": "Monitor berkala tanpa memerlukan intervensi",
    "Medium Risk": "Kirim reminder otomatis",
    "High Risk": "Reconfirmation kepada customer (tetap datang/ubah tanggal/batalkan)"
}

LEAKAGE_FLAGGED_COLS = ["deposit_type", "room_type_changed"]

CURRENCY = "€"

## **0.1 Import Libraries**

In [3]:
import numpy as np
import os
import glob
import joblib
import pandas as pd

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix

## **0.2 Load Model Final**

In [4]:
model_files = glob.glob(os.path.join(MODEL_DIR, "*.pkl"))
print("File model ditemukan:")
for f in model_files:
    print(" -", f)

assert len(model_files) >= 1, f"Tidak ada file .pkl di {MODEL_DIR}. "

MODEL_PATH = model_files[0]
print("\nModel yang dipakai:", MODEL_PATH)

final_model = joblib.load(MODEL_PATH)
final_model

File model ditemukan:
 - ../final_model/XGBoost.pkl

Model yang dipakai: ../final_model/XGBoost.pkl


,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('normal_numeric_pipeline', ...), ('skewed_numeric_pipeline', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## **0.3 Load Testing Datasets**

In [5]:
df_booking = pd.read_csv(DATA_CSV)
df_booking

,Unnamed: 0,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,has_babies,is_family,total_stay_nights,has_agent,has_company,total_previous_bookings,previous_cancellation_rate,has_previous_cancellation,has_booking_changes,room_type_changed
0,9767,Resort Hotel,1,74,2017,January,1,1,1,0,...,0,0,1,1,0,0,0.0,0,0,0
1,9768,Resort Hotel,1,62,2017,January,1,1,2,2,...,0,0,4,1,0,0,0.0,0,0,0
2,9769,Resort Hotel,1,62,2017,January,1,1,2,2,...,0,0,4,1,0,0,0.0,0,0,0
3,9770,Resort Hotel,1,62,2017,January,1,1,2,2,...,0,0,4,1,0,0,0.0,0,0,0
4,9771,Resort Hotel,1,71,2017,January,1,1,2,2,...,0,0,4,1,0,0,0.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40614,119200,City Hotel,0,23,2017,August,35,30,2,5,...,0,0,7,1,0,0,0.0,0,0,0
40615,119201,City Hotel,0,102,2017,August,35,31,2,5,...,0,0,7,1,0,0,0.0,0,0,0
40616,119202,City Hotel,0,34,2017,August,35,31,2,5,...,0,0,7,1,0,0,0.0,0,0,0
40617,119203,City Hotel,0,109,2017,August,35,31,2,5,...,0,0,7,1,0,0,0.0,0,0,0


In [6]:
# 'Unnamed: 0' adalah sisa index dari export CSV sebelumnya, bukan fitur asli —
# drop supaya tidak ikut jadi input model.
STRAY_COLS = [c for c in ["Unnamed: 0"] if c in df_booking.columns]
if STRAY_COLS:
    print(f"Membuang kolom non-fitur: {STRAY_COLS}")

x_booking = df_booking.drop(columns=["is_canceled"] + STRAY_COLS)
y_booking = df_booking["is_canceled"]

Membuang kolom non-fitur: ['Unnamed: 0']


## **0.4 Feature Alignment & Leakage Visibility Check**

In [7]:
# Bandingkan kolom x_booking dengan fitur yang benar-benar dipakai model (kalau tersedia),
# supaya kolom index/kolom asing tidak lolos, dan supaya jelas mana yang sudah expected
# oleh model vs yang belum pernah divalidasi.
expected_features = None
for attr in ("feature_names_in_", "feature_name_"):
    if hasattr(final_model, attr):
        expected_features = list(getattr(final_model, attr))
        break
if expected_features is None:
    try:
        expected_features = list(final_model.get_booster().feature_names)
    except Exception:
        expected_features = None

if expected_features is not None:
    extra_cols = [c for c in x_booking.columns if c not in expected_features]
    missing_cols = [c for c in expected_features if c not in x_booking.columns]
    print("Kolom di x_booking tapi TIDAK dipakai model:", extra_cols or "(tidak ada)")
    print("Kolom yang DIBUTUHKAN model tapi TIDAK ADA di x_booking:", missing_cols or "(tidak ada)")
    assert not missing_cols, "Ada fitur yang dibutuhkan model tapi hilang dari data test — cek pipeline preprocessing."
else:
    print("Tidak bisa membaca daftar fitur model secara otomatis (model bukan estimator sklearn/xgboost standar).")
    print("Lewati alignment check otomatis — cek manual dengan tim yang melatih model.")

# Warning eksplisit untuk kolom yang pernah ditandai leakage risk, TERLEPAS dari
# apakah model memang dilatih dengan kolom itu atau tidak.
present_leakage_cols = [c for c in LEAKAGE_FLAGGED_COLS if c in x_booking.columns]
if present_leakage_cols:
    print(f"\n⚠️  PERINGATAN: kolom berikut pernah ditandai berisiko data leakage "
          f"dan MASIH ada di data test: {present_leakage_cols}")
    print("   Jika belum dikonfirmasi aman (bukan leakage) di notebook feature engineering,")
    print("   metrik performa di notebook ini kemungkinan overestimate kemampuan model.")


Kolom di x_booking tapi TIDAK dipakai model: (tidak ada)
Kolom yang DIBUTUHKAN model tapi TIDAK ADA di x_booking: (tidak ada)

⚠️  PERINGATAN: kolom berikut pernah ditandai berisiko data leakage dan MASIH ada di data test: ['deposit_type', 'room_type_changed']
   Jika belum dikonfirmasi aman (bukan leakage) di notebook feature engineering,
   metrik performa di notebook ini kemungkinan overestimate kemampuan model.


# **Section 1. Model Implementation**

## **1.1 Input to the Model and Classifier Cancel**

In [8]:
cancel_proba = final_model.predict_proba(x_booking)[:, 1]

predicted_cancel = (cancel_proba >= BEST_THRESHOLD).astype(int)

risk_label = pd.cut(cancel_proba, bins=RISK_BINS, labels=RISK_LABELS, right=False)

# RISK_BINS dibangun langsung dari BEST_THRESHOLD (lihat Section 0), jadi "High Risk"
# harus persis sama dengan predicted_cancel == 1. Assert ini menjaga supaya kedua sistem
# labeling tidak kembali tidak sinkron kalau salah satu konstanta diubah di kemudian hari.
assert ((risk_label == "High Risk") == (predicted_cancel == 1)).all(), (
    "RISK_BINS tidak sinkron dengan BEST_THRESHOLD - kategori 'High Risk' harus persis "
    "sama dengan predicted_cancel == 1."
)

print(f"Booking diprediksi CANCEL  : {predicted_cancel.sum():,} / {len(predicted_cancel):,} "
      f"({predicted_cancel.mean():.1%})")
print(f"Booking diprediksi TIDAK cancel : {(predicted_cancel == 0).sum():,}")

print("\nDistribusi risk tier (Low/Medium/High Risk):")
print(risk_label.value_counts().reindex(RISK_LABELS))


Booking diprediksi CANCEL  : 12,237 / 40,619 (30.1%)
Booking diprediksi TIDAK cancel : 28,382

Distribusi risk tier (Low/Medium/High Risk):
Low Risk       14544
Medium Risk    13838
High Risk      12237
Name: count, dtype: int64


In [9]:
y_true = df_booking["is_canceled"]
tn, fp, fn, tp = confusion_matrix(y_true, predicted_cancel, labels=[0, 1]).ravel()

print("Sanity check terhadap data testing:")
print(f"  ROC-AUC   : {roc_auc_score(y_true, cancel_proba):.6f}")
print(f"  Accuracy  : {accuracy_score(y_true, predicted_cancel):.6f}")
print(f"  Precision : {precision_score(y_true, predicted_cancel):.6f}")
print(f"  Recall    : {recall_score(y_true, predicted_cancel):.6f}")
print(f"  F1        : {f1_score(y_true, predicted_cancel):.6f}")
print(f"  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")

Sanity check terhadap data testing:
  ROC-AUC   : 0.886132
  Accuracy  : 0.799232
  Precision : 0.809676
  Recall    : 0.629719
  F1        : 0.708448
  TN=22,556  FP=2,329  FN=5,826  TP=9,908


## **1.2 Check Refund Policy**

In [10]:
def get_recommended_action(risk_label):
    return ACTION_MAP.get(risk_label, "Unknown risk level")

output_df = x_booking.copy().reset_index(drop=True)
output_df["cancel_proba"] = cancel_proba
output_df["risk_label"] = risk_label.astype(str)
output_df["predicted_cancel"] = predicted_cancel
output_df["recommended_action"] = [
    get_recommended_action(risk)
    for risk in output_df["risk_label"]
]

output_df["actual_is_canceled"] = df_booking["is_canceled"].values


In [11]:
display_cols = ["cancel_proba", "risk_label", "predicted_cancel", "recommended_action"]

print("Contoh output:")
display(output_df[display_cols].head(20))

Contoh output:


,cancel_proba,risk_label,predicted_cancel,recommended_action
0,0.822963,High Risk,1,Reconfirmation kepada customer (tetap datang/u...
1,0.481207,Medium Risk,0,Kirim reminder otomatis
2,0.481207,Medium Risk,0,Kirim reminder otomatis
3,0.481207,Medium Risk,0,Kirim reminder otomatis
4,0.766259,High Risk,1,Reconfirmation kepada customer (tetap datang/u...
5,0.448633,Medium Risk,0,Kirim reminder otomatis
6,0.860750,High Risk,1,Reconfirmation kepada customer (tetap datang/u...
7,0.863240,High Risk,1,Reconfirmation kepada customer (tetap datang/u...
8,0.856880,High Risk,1,Reconfirmation kepada customer (tetap datang/u...
9,0.757065,High Risk,1,Reconfirmation kepada customer (tetap datang/u...


In [12]:
print("\n\nDistribusi risk tier x recommended_action:\n")
print(output_df.groupby("risk_label", observed=True)["recommended_action"].value_counts())



Distribusi risk tier x recommended_action:

risk_label   recommended_action                                                 
High Risk    Reconfirmation kepada customer (tetap datang/ubah tanggal/batalkan)    12237
Low Risk     Monitor berkala tanpa memerlukan intervensi                            14544
Medium Risk  Kirim reminder otomatis                                                13838
Name: count, dtype: int64


In [13]:
os.makedirs("../reports/", exist_ok=True)
output_path = "../reports/booking_cancellation_predictions.csv"
output_df.to_csv(output_path, index=False)
print(f"Hasil disimpan ke: {output_path}")

Hasil disimpan ke: ../reports/booking_cancellation_predictions.csv


# **Section 2. Model Limitation**

## **2.1 Data Limitation**

- Dataset bersumber dari **dua hotel** di Portugal, yaitu Resort Hotel di Algarve dan City Hotel di Lisbon, dengan periode pengamatan 2015–2017 (Antonio et al., 2019). Oleh karena itu, generalisasi model ke hotel lain, pasar yang berbeda, atau periode setelah 2017 **tidak dapat dijamin**. Perubahan perilaku booking setelah periode tersebut, termasuk perubahan struktural pasca-pandemi, juga belum tercakup dalam data training model.
- Beberapa kategori memiliki jumlah observasi yang sangat kecil. Cancellation rate pada kategori tersebut dapat terlihat ekstrem, tetapi memiliki tingkat ketidakpastian yang tinggi sehingga **tidak sebaiknya digunakan sebagai dasar keputusan bisnis** tanpa mempertimbangkan ukuran sampelnya.
- Evaluasi berdasarkan split temporal dengan **train = 2015, validation = 2016, dan test = 2017** menunjukkan penurunan ROC-AUC secara monoton, yaitu **0.9670 (2015) → 0.9352 (2016) → 0.8861 (2017)**. Pola ini mengindikasikan adanya komponen *concept drift*, yaitu perubahan pola hubungan antara karakteristik reservasi dan cancellation dari waktu ke waktu. Oleh karena itu, model ini **bukan solusi *deploy-and-forget*** dan memerlukan monitoring performa serta jadwal retraining secara berkala.

## **2.2 Risk of Data Leakage**

- `room_type_changed`: `assigned_room_type` pada data historis berpotensi baru difinalisasi mendekati waktu check-in. Penggunaan fitur ini berisiko memasukkan informasi yang belum tersedia pada titik keputusan bisnis ketika prediksi cancellation seharusnya dilakukan.
- `deposit_type = "Non Refund"` memiliki cancellation rate **99,4%** pada data historis, jauh lebih tinggi dibandingkan `No Deposit` sebesar **28,4%** dan `Refundable` sebesar **22,2%**. Meskipun sangat prediktif, fitur ini berisiko menjadi *dominant proxy* yang membuat performa model banyak ditopang oleh satu informasi yang secara bisnis dapat berkaitan erat dengan keputusan atau karakteristik reservasi, bukan semata-mata dengan faktor independen yang menyebabkan cancellation.
- `previous_cancellations` menunjukkan pola yang divergen pada metode *feature importance* yang diuji. Fitur ini memiliki peringkat relatif tinggi pada *gain-based* `feature_importances_` dan SHAP yang dihitung pada `x_train`, tetapi kontribusinya hampir nol pada *Permutation Importance* yang dihitung pada `x_test` secara *out-of-sample*. Pola bahwa suatu fitur terlihat penting pada data training tetapi kehilangan kontribusi pada data test konsisten dengan kemungkinan adanya **leakage atau sinyal yang tidak mampu digeneralisasikan**, sehingga fitur ini perlu diinvestigasi lebih lanjut sebelum digunakan sebagai *business driver*.

## **2.3 Limitations of Feature Importance Interpretation**

- Tiga metode *feature importance*, yaitu *gain-based* `feature_importances_`, SHAP, dan *Permutation Importance*, **tidak sepenuhnya menghasilkan urutan fitur yang sama**. Perbedaan paling mencolok terlihat pada `agent`, yang di-encode menggunakan `TargetEncoder`. Fitur ini berada pada peringkat **#7** berdasarkan *gain-based importance*, tetapi berada pada peringkat **#1** berdasarkan SHAP dan *Permutation Importance*. Perbedaan tersebut menunjukkan bahwa *gain-based importance* dapat memberikan estimasi kontribusi yang berbeda dibandingkan metode yang mengukur dampak fitur terhadap prediksi atau performa model.
- SHAP pada analisis interpretasi dihitung menggunakan sampel `x_train`, sedangkan *Permutation Importance* dihitung menggunakan `x_test` yang merepresentasikan tahun 2017. Karena kedua metode dihitung pada populasi data yang berbeda, perbedaan ranking tidak hanya dapat disebabkan oleh perbedaan metode, tetapi juga oleh **pergeseran karakteristik data antarperiode**. Perbandingan yang lebih konsisten idealnya dilakukan dengan menghitung SHAP dan *Permutation Importance* pada dataset evaluasi yang sama, misalnya `x_test`.
- `agent` merupakan fitur kategorikal dengan jumlah level unik yang tinggi dan diubah menjadi satu kolom numerik menggunakan `TargetEncoder`. Importance yang tinggi pada fitur dengan karakteristik tersebut perlu diinterpretasikan secara hati-hati karena kontribusinya dapat dipengaruhi oleh distribusi booking pada masing-masing agent, terutama agent dengan jumlah observasi yang relatif kecil. Oleh karena itu, importance `agent` secara keseluruhan belum cukup untuk menghasilkan insight bisnis yang actionable pada tingkat masing-masing agent tanpa analisis tambahan.

## **2.4 Assumptions in Cost-Benefit Simulations**

- Biaya intervensi (`COST_PER_CONTACT` = €1,11) dibangun dari tiga asumsi manual, yaitu `EMPLOYEE_COST_PER_MINUTE` (€0,17/menit), `PHONE_COST_PER_MINUTE` (€0,20/menit), dan `CONTACT_DURATION_MINUTES` (3 menit), bukan berdasarkan hasil pengukuran biaya operasional aktual. Namun, hasil simulasi menunjukkan *break-even success rate* hanya sebesar **0,32%**. Artinya, kesimpulan bahwa intervensi berpotensi menguntungkan secara ekonomi relatif **tidak sensitif terhadap ketidakakuratan ketiga asumsi biaya tersebut**, karena tingkat keberhasilan minimum yang diperlukan untuk mencapai *break-even* jauh lebih rendah dibandingkan skenario *success rate* yang diuji.

- Sebaliknya, hasil simulasi jauh lebih sensitif terhadap dua asumsi yang belum divalidasi, yaitu: **(1)** `SUCCESS_RATES` sebesar 30%, 50%, dan 70% sebagai probabilitas intervensi berhasil mencegah cancellation, yang masih merupakan asumsi skenario dan belum didukung oleh *pilot test* atau data historis intervensi; serta **(2)** *potential loss* yang dihitung menggunakan `adr * total_stay_nights`, tanpa memperhitungkan kemungkinan kamar berhasil dijual kembali (*reselling*), variasi harga akibat *seasonality*, maupun kemungkinan tamu melakukan *reschedule* alih-alih membatalkan reservasi sepenuhnya. Oleh karena itu, nilai kerugian yang digunakan dalam simulasi sebaiknya dipandang sebagai **estimasi potensi kehilangan pendapatan**, bukan sebagai kerugian aktual yang pasti terjadi.

- Dengan demikian, hasil *cost-benefit analysis* lebih tepat digunakan sebagai **indikasi awal kelayakan ekonomi**, bukan sebagai estimasi final atas penghematan yang akan diperoleh. Dua komponen yang paling menentukan, yaitu *success rate* intervensi dan estimasi *potential loss*, perlu divalidasi melalui data operasional atau *pilot implementation* bersama tim bisnis maupun Customer Service sebelum simulasi digunakan sebagai dasar keputusan investasi atau implementasi intervensi dalam skala penuh.

## **2.5 Limitations of Generalizing to the Current Situation**

Model dilatih sepenuhnya menggunakan data historis **2015–2017 dari dua hotel di Portugal**. Oleh karena itu, pola booking, saluran distribusi, karakteristik reservasi, dan perilaku cancellation pada kondisi saat ini kemungkinan berbeda dari pola yang dipelajari model. Performa pada data produksi saat ini **tidak dapat diasumsikan setara** dengan performa pada test set 2017 yang memiliki ROC-AUC sebesar **0.8861**.

Selain itu, penurunan ROC-AUC secara temporal dari **0.9670 (2015) → 0.9352 (2016) → 0.8861 (2017)** menunjukkan adanya indikasi perubahan pola dari waktu ke waktu. Dengan demikian, sebelum model diterapkan pada kondisi aktual, diperlukan **validasi menggunakan data terbaru** untuk memastikan bahwa sinyal prediktif yang dipelajari masih relevan. Setelah deployment, monitoring performa dan karakteristik input perlu dilakukan secara berkala, disertai **retraining menggunakan data terbaru** apabila terjadi degradasi performa atau perubahan pola booking dan cancellation yang signifikan.

# **Section 3. Cost Benefit Analysis**

In [14]:
# Biaya tenaga kerja + biaya telepon
COST_PER_CONTACT = (EMPLOYEE_COST_PER_MINUTE + PHONE_COST_PER_MINUTE) * CONTACT_DURATION_MINUTES

y_true_array = np.asarray(y_true).astype(int).ravel()

predicted_cancel_array = (np.asarray(predicted_cancel).astype(int).ravel())

cancel_proba_array = (np.asarray(cancel_proba).astype(float).ravel())

# Nilai bruto reservasi yang berisiko hilang
booking_value_at_risk = (x_booking["adr"] * x_booking["total_stay_nights"]).reset_index(drop=True).to_numpy(dtype=float)

data_lengths = {
    "booking_value_at_risk": len(booking_value_at_risk),
    "y_true": len(y_true_array),
    "predicted_cancel": len(predicted_cancel_array),
    "cancel_proba": len(cancel_proba_array)
}

tn, fp, fn, tp = confusion_matrix(y_true_array, predicted_cancel_array, labels=[0, 1]).ravel()

is_tp = ((y_true_array == 1) & (predicted_cancel_array == 1))
is_fp = ((y_true_array == 0) & (predicted_cancel_array == 1))
is_fn = ((y_true_array == 1) & (predicted_cancel_array == 0))
is_tn = ((y_true_array == 0) & (predicted_cancel_array == 0))

# Total nilai reservasi dari semua booking yang benar-benar cancel
total_booking_value_at_risk = (booking_value_at_risk[y_true_array == 1].sum())

# Nilai reservasi dari cancellation yang berhasil dideteksi model
tp_booking_value_at_risk = (booking_value_at_risk[is_tp].sum())

# Semua booking yang diprediksi cancel akan dihubungi
total_contacted = int(is_tp.sum() + is_fp.sum())

# Total biaya menghubungi TP dan FP
total_intervention_cost = (total_contacted * COST_PER_CONTACT)

# Tingkat keberhasilan minimum agar biaya intervensi tertutupi
if tp_booking_value_at_risk > 0:
    break_even_success_rate = (total_intervention_cost / tp_booking_value_at_risk)
else:
    break_even_success_rate = np.nan

def cost_benefit(success_rate: float) -> dict:
    # Nilai reservasi yang diperkirakan berhasil dipertahankan
    protected_revenue = (tp_booking_value_at_risk * success_rate)

    # Nilai reservasi yang masih berisiko hilang
    remaining_revenue_at_risk = (total_booking_value_at_risk - protected_revenue)

    # Total dampak biaya setelah model digunakan
    total_cost_after_model = (remaining_revenue_at_risk + total_intervention_cost)

    # Nilai reservasi yang dipertahankan setelah biaya intervensi
    net_revenue_protected = (protected_revenue - total_intervention_cost)

    # Return on intervention cost
    if total_intervention_cost > 0:
        intervention_roi = (net_revenue_protected / total_intervention_cost)
    else:
        intervention_roi = np.nan

    return {
        "success_rate": success_rate,
        "contacts": total_contacted,
        "cost_per_contact": COST_PER_CONTACT,
        "intervention_cost": total_intervention_cost,
        "protected_revenue": protected_revenue,
        "remaining_revenue_at_risk": remaining_revenue_at_risk,
        "total_cost_after_model": total_cost_after_model,
        "net_revenue_protected": net_revenue_protected,
        "intervention_roi": intervention_roi,
        "break_even_success_rate": break_even_success_rate
    }

separator = "=" * 86

print(f"\n{separator}")
print("SIMULASI DAMPAK BISNIS — HOTEL BOOKING CANCELLATION")
print(separator)

print("\nMODEL PERFORMANCE")
print(f"Total booking                      : {len(y_true_array):,}")
print(f"Booking cancel sebenarnya          : {(tp + fn):,}")
print(f"Booking diprediksi cancel          : {(tp + fp):,}")
print(f"True Positive                      : {tp:,}")
print(f"False Positive                     : {fp:,}")
print(f"False Negative                     : {fn:,}")
print(f"True Negative                      : {tn:,}")

print(f"Accuracy                           : {accuracy_score(y_true_array, predicted_cancel_array):.4f}")
print(f"Precision                          : {precision_score(y_true_array, predicted_cancel_array, zero_division=0):.4f}")
print(f"Recall                             : {recall_score(y_true_array, predicted_cancel_array, zero_division=0):.4f}")
print(f"F1-score                           : {f1_score(y_true_array, predicted_cancel_array, zero_division=0):.4f}")
print(f"ROC-AUC                            : {roc_auc_score(y_true_array, cancel_proba_array):.4f}")

print("\nASUMSI BIAYA INTERVENSI")
print(f"Biaya waktu staf per menit         : {CURRENCY} {EMPLOYEE_COST_PER_MINUTE:.2f}")
print(f"Biaya telepon per menit            : {CURRENCY} {PHONE_COST_PER_MINUTE:.2f}")
print(f"Durasi setiap panggilan            : {CONTACT_DURATION_MINUTES} menit")
print(f"Biaya setiap booking yang dihubungi: {CURRENCY} {COST_PER_CONTACT:.2f}")
print(f"Jumlah booking yang dihubungi      : {total_contacted:,}")

print(f"Total biaya intervensi             : {CURRENCY} {total_intervention_cost:,.2f}")

print("\nTANPA MODEL")
print(f"Total booking value at risk        : {CURRENCY} {total_booking_value_at_risk:,.2f}")

print("\nDENGAN MODEL")
print(
    f"{'Success':>9} | "
    f"{'Revenue protected':>20} | "
    f"{'Revenue masih berisiko':>23} | "
    f"{'Biaya intervensi':>18} | "
    f"{'Net protected':>18}"
)

print("-" * 101)

for rate in SUCCESS_RATES:
    result = cost_benefit(rate)

    print(
        f"{rate:>9.0%} | "
        f"{CURRENCY} {result['protected_revenue']:>18,.0f} | "
        f"{CURRENCY} {result['remaining_revenue_at_risk']:>21,.0f} | "
        f"{CURRENCY} {result['intervention_cost']:>16,.0f} | "
        f"{CURRENCY} {result['net_revenue_protected']:>16,.0f}"
    )

print(f"\n{separator}")

if np.isnan(break_even_success_rate):
    print("Break-even success rate           : Tidak dapat dihitung karena tidak terdapat nilai TP.")
else:
    print(f"Break-even success rate           : {break_even_success_rate:.2%}")

print(f"Biaya intervensi per booking      : {CURRENCY} {COST_PER_CONTACT:.2f}")

print("\nCatatan: hasil menunjukkan estimasi revenue yang dapat dipertahankan, bukan keuntungan bersih hotel.")

print("ADR × total stay nights masih merupakan nilai bruto reservasi dan belum memperhitungkan kemungkinan kamar dijual kembali maupun biaya operasional hotel.")


SIMULASI DAMPAK BISNIS — HOTEL BOOKING CANCELLATION

MODEL PERFORMANCE
Total booking                      : 40,619
Booking cancel sebenarnya          : 15,734
Booking diprediksi cancel          : 12,237
True Positive                      : 9,908
False Positive                     : 2,329
False Negative                     : 5,826
True Negative                      : 22,556
Accuracy                           : 0.7992
Precision                          : 0.8097
Recall                             : 0.6297
F1-score                           : 0.7084
ROC-AUC                            : 0.8861

ASUMSI BIAYA INTERVENSI
Biaya waktu staf per menit         : € 0.17
Biaya telepon per menit            : € 0.20
Durasi setiap panggilan            : 3 menit
Biaya setiap booking yang dihubungi: € 1.11
Jumlah booking yang dihubungi      : 12,237
Total biaya intervensi             : € 13,583.07

TANPA MODEL
Total booking value at risk        : € 7,223,579.90

DENGAN MODEL
  Success |    Revenue protec